# Used Car Price Prediction
## Notebook 02 — Preprocessing

**Goal:** Apply the cleaning and feature engineering pipeline from `src/preprocessing.py` and document every decision before model selection.

**Input:** `cardekho_dataset.csv` (raw)

**Outputs:**
- `outputs/preprocessed/cleaned_dataset.csv`
- `outputs/preprocessed/modeling_dataset.csv`

**Sections:**
1. Environment setup
2. Load raw data
3. Run preprocessing pipeline
4. Before/after comparison
5. Validity filtering report
6. Outlier flags inspection
7. Rare category grouping
8. Engineered features check
9. Final modeling dataset
10. Outputs verification
11. Findings summary

## 1. Environment setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from preprocessing import (
    load_raw_data,
    clean_data,
    flag_outliers,
    group_rare_categories,
    engineer_features,
    build_model_frame,
    build_preprocessor,
    save_outputs,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110

PREPROCESSED_DIR = PROJECT_ROOT / 'outputs' / 'preprocessed'
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Preprocessed dir:', PREPROCESSED_DIR)

## 2. Load raw data

In [ ]:
raw = load_raw_data()
raw.head()

## 3. Run preprocessing pipeline

The pipeline is applied step by step so that the effect of each stage can be inspected.

In [ ]:
cleaned = clean_data(raw)
flagged = flag_outliers(cleaned)
grouped = group_rare_categories(flagged)
engineered = engineer_features(grouped)
modeling = build_model_frame(engineered)
save_outputs(engineered, modeling)

## 4. Before / after comparison

In [ ]:
comparison = pd.DataFrame({
    'stage': ['raw', 'after clean', 'after grouping', 'after engineering', 'modeling dataset'],
    'rows': [len(raw), len(cleaned), len(grouped), len(engineered), len(modeling)],
    'columns': [raw.shape[1], cleaned.shape[1], grouped.shape[1], engineered.shape[1], modeling.shape[1]],
})
comparison['rows_dropped'] = comparison['rows'].shift(1) - comparison['rows']
comparison

## 5. Validity filtering report

Rows removed by business-rule checks in `clean_data`.

In [ ]:
rules = {
    'negative_vehicle_age': raw['vehicle_age'] < 0,
    'negative_km_driven': raw['km_driven'] < 0,
    'nonpositive_mileage': raw['mileage'] <= 0,
    'nonpositive_engine': raw['engine'] <= 0,
    'nonpositive_max_power': raw['max_power'] <= 0,
    'seats_outside_2_10': ~raw['seats'].between(2, 10),
    'nonpositive_selling_price': raw[TARGET] <= 0,
}
validity = pd.DataFrame([
    {'rule': k, 'flagged_count': int(v.sum()), 'flagged_percent': round(v.mean() * 100, 3)}
    for k, v in rules.items()
])
validity.to_csv(PREPROCESSED_DIR / 'validity_report.csv', index=False)
validity

## 6. Outlier flags inspection

Outliers are flagged, not removed. Tree-based models are generally robust to outliers; linear models can downweight them later via `sample_weight`.

In [ ]:
outlier_flags = ['km_driven_is_outlier', 'selling_price_is_outlier']
flag_summary = pd.DataFrame({
    'flag': outlier_flags,
    'count': [int(flagged[c].sum()) for c in outlier_flags],
    'percent': [round(flagged[c].mean() * 100, 2) for c in outlier_flags],
})
flag_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(
    data=flagged, x='selling_price', hue='selling_price_is_outlier',
    bins=50, ax=axes[0], palette={0: '#4C72B0', 1: '#C44E52'},
)
axes[0].set_title('Selling price with outliers flagged')

sns.histplot(
    data=flagged, x='km_driven', hue='km_driven_is_outlier',
    bins=50, ax=axes[1], palette={0: '#4C72B0', 1: '#C44E52'},
)
axes[1].set_title('km_driven with outliers flagged')

plt.tight_layout()
plt.show()

## 7. Rare category grouping

Categories with fewer than `MIN_CATEGORY_COUNT` occurrences are grouped into `__rare__` for `brand` and `model`.

In [ ]:
for col in ['brand', 'model']:
    before = cleaned[col].nunique()
    after = grouped[col].nunique()
    rare_rows = int((grouped[col] == '__rare__').sum())
    print(f'{col}: {before} -> {after} unique values | rows in __rare__: {rare_rows}')

## 8. Engineered features check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(engineered['km_per_year'].dropna(), bins=50, ax=axes[0])
axes[0].set_title('km_per_year')

sns.histplot(engineered['power_per_engine'].dropna(), bins=50, ax=axes[1])
axes[1].set_title('power_per_engine')

engineered['brand_tier'].value_counts().plot(kind='bar', ax=axes[2])
axes[2].set_title('brand_tier distribution')

plt.tight_layout()
plt.show()

In [ ]:
engineered[['km_per_year', 'power_per_engine']].describe().T

## 9. Final modeling dataset

The modeling dataset contains only the features used during training plus the target.

In [ ]:
print('Shape:', modeling.shape)
print('Numeric features:', NUMERIC_FEATURES)
print('Categorical features:', CATEGORICAL_FEATURES)
print('Target:', TARGET)
print('Missing values:', int(modeling.isna().sum().sum()))
modeling.head()

In [ ]:
modeling.dtypes

## 10. Outputs verification

In [ ]:
for name in ['cleaned_dataset.csv', 'modeling_dataset.csv']:
    path = PREPROCESSED_DIR / name
    exists = path.exists()
    size_kb = round(path.stat().st_size / 1024, 1) if exists else 0
    print(f'{name}: exists={exists} size={size_kb} KB')

In [ ]:
loaded = pd.read_csv(PREPROCESSED_DIR / 'modeling_dataset.csv')
print('Loaded modeling dataset shape:', loaded.shape)
print('Matches expected shape:', loaded.shape == modeling.shape)
loaded.head()

In [ ]:
preprocessor = build_preprocessor()
print('Preprocessor type:', type(preprocessor).__name__)
print('Numeric transformers:', [t[0] for t in preprocessor.transformers_ if t[0] == 'num'])
print('Categorical transformers:', [t[0] for t in preprocessor.transformers_ if t[0] == 'cat'])
preprocessor

## 11. Findings summary

In [ ]:
summary = pd.DataFrame([
    {'decision': 'Drop Unnamed index and car_name', 'rows_affected': None,
     'rationale': 'No information content'},
    {'decision': 'Normalize model names', 'rows_affected': None,
     'rationale': 'RediGO / redi-GO, Dzire variants unified'},
    {'decision': 'Drop rows failing validity rules',
     'rows_affected': len(raw) - len(cleaned) - int(raw.duplicated().sum()),
     'rationale': 'Non-positive values, seats outside 2-10'},
    {'decision': 'Drop exact duplicates',
     'rows_affected': int(raw.duplicated().sum()),
     'rationale': 'Prevent duplication bias'},
    {'decision': 'Flag km_driven / selling_price outliers',
     'rows_affected': int(flagged['km_driven_is_outlier'].sum() + flagged['selling_price_is_outlier'].sum()),
     'rationale': 'Kept for tree-based models; can be down-weighted for linear models'},
    {'decision': 'Group rare brand / model values',
     'rows_affected': int((grouped['brand'] == '__rare__').sum() + (grouped['model'] == '__rare__').sum()),
     'rationale': 'Reduce cardinality, improve generalization'},
    {'decision': 'Add km_per_year and power_per_engine', 'rows_affected': None,
     'rationale': 'Stronger signal than raw km_driven / max_power'},
    {'decision': 'Add brand_tier (domain mapping)', 'rows_affected': None,
     'rationale': 'Avoid target leakage; encode market segment'},
    {'decision': 'Final modeling dataset', 'rows_affected': len(modeling),
     'rationale': f'{modeling.shape[1]} columns ready for split and model selection'},
])
summary

In [ ]:
summary.to_csv(PREPROCESSED_DIR / 'preprocessing_summary.csv', index=False)
print('Summary saved to', (PREPROCESSED_DIR / 'preprocessing_summary.csv').resolve())

### Next steps

- `notebooks/03_model_selection.ipynb` will split `modeling_dataset.csv` into train/test, fit the pipeline from `build_preprocessor()`, and compare candidate models with cross-validation on the training set only.
- The split remains outside this notebook on purpose so that the modeling dataset stays reusable for different split strategies (random, stratified by price bins, time-based, etc.).